In [3]:
%matplotlib inline

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.io import read_image
import os
from PIL import Image
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import cv2

In [5]:
import os
import shutil
import random

# Define dataset paths
folder_path = "D:/sqldev/RetinalOCT_Dataset"

train_dir = os.path.join(folder_path, "D:/sqldev/RetinalOCT_Dataset/train")
val_dir = os.path.join(folder_path, "D:/sqldev/RetinalOCT_Dataset/val")
test_dir = os.path.join(folder_path, "D:/sqldev/RetinalOCT_Dataset/test")

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [7]:





# Transformations with augmentation for training
train_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

val_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])


In [8]:

import torch
from torch.utils.data import Dataset
import os
from torchvision.io import read_image
import torchvision.transforms as T

class CustomImageDataset(Dataset):
    def __init__(self, ds_path, transforms=None, target_transform=None):
        self.ds_path = ds_path
        self.transforms = transforms
        self.target_transform = target_transform
        self.labels = os.listdir(ds_path)  # Get list of label folder names
        self.img_paths = []

        # Create a list of image paths and ensure all labels are mapped correctly
        for label in self.labels:
            base_path = os.path.join(ds_path, label)
            if os.path.isdir(base_path):  # Ensure it's a directory
                for img in os.listdir(base_path):
                    img_path = os.path.join(base_path, img)
                    self.img_paths.append(img_path)

        # Mapping labels to numeric values
        self.label_map = {
            'AMD': 0,
            'CNV': 1,
            'CSR': 2,
            'DME': 3,
            'DR': 4,
            'DRUSEN': 5,
            'MH': 6,
            'NORMAL': 7
        }

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        image = read_image(img_path)

        # Extract the label from the folder name in a cross-platform way
        label = os.path.basename(os.path.dirname(img_path))

        # Ensure label exists in label_map, otherwise raise an error
        if label not in self.label_map:
            raise KeyError(f"Label '{label}' not found in label_map. Check dataset structure.")

        image = image.float() / 255.0  # Normalize pixel values to [0,1]

        # Convert grayscale images to RGB if needed
        if image.shape[0] == 1:
            image = image.repeat(3, 1, 1)

        # Apply transformations
        if self.transforms:
            image = self.transforms(image)

        image = T.Resize((224, 224))(image)  # Ensure image size is consistent

        return image, self.label_map[label]




In [9]:
import torch
import torch.nn as nn
import torchvision.models as models

class ResNet50LSTM(nn.Module):
    def __init__(self, num_classes=8, lstm_hidden_size=256, lstm_num_layers=4):
        super(ResNet50LSTM, self).__init__()

        # Load pretrained ResNet-50
        resnet = models.resnet50(pretrained=True)

        # Remove the average pool and fully connected layers
        self.feature_extractor = nn.Sequential(*list(resnet.children())[:-2])  # Output: [B, 2048, 7, 7]

        # LSTM over spatial patches
        self.lstm = nn.LSTM(
            input_size=2048,            # ResNet-50 final feature dim is 2048
            hidden_size=lstm_hidden_size,
            num_layers=lstm_num_layers,
            batch_first=True
        )

        # Final classification layer
        self.fc = nn.Linear(lstm_hidden_size, num_classes)

    def forward(self, x):
        # Feature map extraction
        x = self.feature_extractor(x)  # [B, 2048, 7, 7]

        # Convert to sequence of 49 (7x7) patches
        x = x.view(x.size(0), 2048, -1).permute(0, 2, 1)  # [B, 49, 2048]

        # Pass through LSTM
        lstm_out, _ = self.lstm(x)  # [B, 49, hidden_size]

        # Take last output from sequence
        x = lstm_out[:, -1, :]  # [B, hidden_size]

        # Final prediction
        out = self.fc(x)  # [B, num_classes]
        return out


In [10]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        self.stopped_epoch = 0

    def on_epoch_end(self, epoch, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                self.stopped_epoch = epoch


In [11]:

def train_model(model, criterion, optimizer, scheduler, data_loaders, device, num_epochs, patience=5):
    early_stop = EarlyStopping(patience=patience)
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss, running_corrects, total = 0.0, 0, 0

            for inputs, labels in tqdm(data_loaders[phase]):
                inputs = inputs.to(device)
                labels = labels.to(device)

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        optimizer.zero_grad()
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                total += inputs.size(0)

            epoch_loss = running_loss / total
            epoch_acc = running_corrects.double() / total

            history[f"{phase}_loss"].append(epoch_loss)
            history[f"{phase}_acc"].append(epoch_acc.item())

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'val':
                scheduler.step(epoch_loss)
                early_stop.on_epoch_end(epoch, epoch_loss)
                if early_stop.early_stop:
                    print("Early stopping triggered.")
                    return history

    return history


In [13]:

model = ResNet50LSTM().to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam(model.parameters(), lr=0.0001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)


d:\Python\myenv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\Python\myenv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:

train_dataset = CustomImageDataset('D:/sqldev/RetinalOCT_Dataset/train', transforms=train_transforms)
val_dataset = CustomImageDataset('D:/sqldev/RetinalOCT_Dataset/val', transforms=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

data_loaders = {'train': train_loader, 'val': val_loader}

history = train_model(model, criterion, optimizer, scheduler, data_loaders, device, num_epochs=5, patience=5)


In [ ]:

def predict_with_tta(model, dataloader, tta_transforms, n_augmentations=5):
    model.eval()
    all_preds = []

    with torch.no_grad():
        for inputs, _ in tqdm(dataloader):
            inputs = inputs.to(device)
            aug_preds = []

            for _ in range(n_augmentations):
                augmented = torch.stack([tta_transforms(img.cpu()).to(device) for img in inputs])
                outputs = model(augmented)
                aug_preds.append(outputs.softmax(dim=1))

            avg_pred = torch.mean(torch.stack(aug_preds), dim=0)
            all_preds.append(avg_pred.cpu())

    return torch.cat(all_preds, dim=0)


In [ ]:
from torchvision.models import resnet18
from torch.nn import functional as F
import cv2

def grad_cam(model, image_tensor, class_idx=None):
    image_tensor = image_tensor.unsqueeze(0).to(device)
    model.eval()

    features = []
    gradients = []

    def forward_hook(module, input, output):
        features.append(output)

    def backward_hook(module, grad_in, grad_out):
        gradients.append(grad_out[0])

    final_conv = model.resnet[-1]
    handle_fw = final_conv.register_forward_hook(forward_hook)
    handle_bw = final_conv.register_backward_hook(backward_hook)

    output = model(image_tensor)
    if class_idx is None:
        class_idx = output.argmax().item()

    model.zero_grad()
    class_score = output[0, class_idx]
    class_score.backward()

    grads = gradients[0]
    fmap = features[0]
    weights = grads.mean(dim=[2, 3], keepdim=True)
    cam = (weights * fmap).sum(dim=1).squeeze()

    cam = F.relu(cam)
    cam = cam - cam.min()
    cam = cam / cam.max()
    cam = cam.cpu().detach().numpy()
    cam = cv2.resize(cam, (224, 224))

    img_np = image_tensor.squeeze().permute(1, 2, 0).cpu().numpy()
    plt.imshow(img_np)
    plt.imshow(cam, cmap='jet', alpha=0.5)
    plt.title(f"Grad-CAM (Class {class_idx})")
    plt.axis("off")
    plt.show()

    handle_fw.remove()
    handle_bw.remove()
